# 00 - Load Dataset: memuat MIRACL id dari berkas mentah

Alur: baca revision unduhan → muat korpus `jsonl.gz` → muat topics dan qrels dev `TSV` → periksa jumlah baris → simpan ke `data/interim/`.

Keluaran di `data/interim/`:
- `corpus.parquet` — kolom `docid`, `title`, `text`
- `topics.parquet` — kolom `query_id`, `text`
- `qrels.parquet` — kolom `query_id`, `iteration`, `doc_id`, `relevance`
- `metadata.json` — repo, revision, berkas sumber, dan jumlah baris tiap tabel

Isi tabel adalah salinan berkas mentah apa adanya; validasi kunci dan relasi dikerjakan `01_preprocessing`.

Prasyarat:
- `pip install -r requirements.txt` (K5: `datasets` 5.0.1).
- Dataset diunduh Arya dari root project dengan `hf download --local-dir`, supaya revision setiap berkas tercatat di `.cache/huggingface/download/`:

```bash
hf download miracl/miracl-corpus --repo-type dataset --include "miracl-corpus-v1.0-id/*" --local-dir data/raw/miracl-corpus
hf download miracl/miracl --repo-type dataset --include "miracl-v1.0-id/topics/*dev*" "miracl-v1.0-id/qrels/*dev*" --local-dir data/raw/miracl
```

Keputusan: D3, K5 (002a); K9 (003a).

In [ ]:
from pathlib import Path

# Folder hasil `hf download --local-dir` untuk korpus
CORPUS_DIR = Path("../data/raw/miracl-corpus")
# Folder hasil `hf download --local-dir` untuk topics dan qrels
MIRACL_DIR = Path("../data/raw/miracl")
# Folder keluaran notebook ini, dibaca 01_preprocessing
INTERIM_DIR = Path("../data/interim")

# Repo Hugging Face asal korpus, dicatat di metadata.json
CORPUS_REPO = "miracl/miracl-corpus"
# Repo Hugging Face asal topics dan qrels, dicatat di metadata.json
MIRACL_REPO = "miracl/miracl"

# Pola nama berkas korpus bahasa Indonesia di dalam CORPUS_DIR
CORPUS_GLOB = "miracl-corpus-v1.0-id/docs-*.jsonl.gz"
# Jumlah berkas korpus menurut K5
CORPUS_FILE_COUNT = 3
# Berkas topics split dev di dalam MIRACL_DIR
TOPICS_FILE = "miracl-v1.0-id/topics/topics.miracl-v1.0-id-dev.tsv"
# Berkas qrels split dev di dalam MIRACL_DIR
QRELS_FILE = "miracl-v1.0-id/qrels/qrels.miracl-v1.0-id-dev.tsv"

# Kolom korpus di berkas jsonl
CORPUS_COLUMNS = ["docid", "title", "text"]
# Kolom topics; berkas TSV tidak punya baris judul
TOPICS_COLUMNS = ["query_id", "text"]
# Kolom qrels format TREC: query, iterasi (Q0), dokumen, nilai relevansi
QRELS_COLUMNS = ["query_id", "iteration", "doc_id", "relevance"]

# Jumlah baris menurut D3 (002a)
EXPECTED_CORPUS_ROWS = 1_446_315
EXPECTED_TOPICS_ROWS = 960
EXPECTED_QRELS_ROWS = 9_668

## 1. Revision dataset

Revision dibaca dari metadata unduhan, bukan dari Hub, supaya yang tercatat adalah versi berkas yang benar-benar dimuat (K5). Semua berkas dari satu repo harus berasal dari satu commit.

In [ ]:
import re

# Commit hash Hugging Face: 40 karakter heksadesimal, contoh 0123abcd...89ef
COMMIT_HASH_PATTERN = re.compile(r"[0-9a-f]{40}")


def fetch_corpus_files(corpus_dir: Path, pattern: str, expected_count: int) -> list[Path]:
    """Cari berkas korpus dan pastikan jumlahnya sesuai.

    Args:
        corpus_dir: Folder hasil unduhan korpus.
        pattern: Pola glob relatif terhadap corpus_dir.
        expected_count: Jumlah berkas yang diharapkan.

    Returns:
        Path berkas korpus, terurut menurut nama.

    Raises:
        FileNotFoundError: Jumlah berkas yang ditemukan berbeda dari expected_count.
    """
    files = sorted(corpus_dir.glob(pattern))
    if len(files) != expected_count:
        raise FileNotFoundError(
            f"{len(files)} berkas cocok dengan {corpus_dir / pattern}, seharusnya {expected_count}"
        )
    return files


def fetch_dataset_revision(local_dir: Path, relative_paths: list[str]) -> str:
    """Baca commit hash berkas hasil `hf download --local-dir`.

    Args:
        local_dir: Folder `--local-dir` yang dipakai saat mengunduh.
        relative_paths: Path berkas relatif terhadap local_dir, memakai `/`.

    Returns:
        Commit hash yang sama untuk semua berkas.

    Raises:
        FileNotFoundError: Metadata unduhan sebuah berkas tidak ada.
        ValueError: Metadata kosong, bukan commit hash, atau berkas berasal dari commit berbeda.
    """
    revisions: set[str] = set()
    for relative_path in relative_paths:
        metadata_path = local_dir / ".cache" / "huggingface" / "download" / f"{relative_path}.metadata"
        if not metadata_path.is_file():
            raise FileNotFoundError(
                f"Metadata unduhan tidak ditemukan: {metadata_path}; unduh dengan `hf download --local-dir`"
            )
        lines = metadata_path.read_text(encoding="utf-8").splitlines()
        commit_hash = lines[0].strip() if lines else ""
        if COMMIT_HASH_PATTERN.fullmatch(commit_hash) is None:
            raise ValueError(f"Baris pertama {metadata_path} bukan commit hash: {commit_hash!r}")
        revisions.add(commit_hash)
    if len(revisions) != 1:
        raise ValueError(f"Berkas di {local_dir} berasal dari {len(revisions)} commit: {sorted(revisions)}")
    return revisions.pop()


corpus_files = fetch_corpus_files(CORPUS_DIR, CORPUS_GLOB, CORPUS_FILE_COUNT)
corpus_paths = [file.relative_to(CORPUS_DIR).as_posix() for file in corpus_files]
corpus_revision = fetch_dataset_revision(CORPUS_DIR, corpus_paths)
miracl_revision = fetch_dataset_revision(MIRACL_DIR, [TOPICS_FILE, QRELS_FILE])
print({"corpus_files": corpus_paths, "corpus_revision": corpus_revision, "miracl_revision": miracl_revision})

## 2. Korpus

Dimuat dengan builder `json` dari tiga berkas `jsonl.gz` karena `datasets` 5.x tidak mendukung loading script `miracl.py` (K5). Semua kolom dibaca sebagai teks supaya `docid` tidak berubah bentuk.

In [ ]:
from datasets import Dataset, Features, Value, load_dataset


def load_corpus(files: list[Path], columns: list[str]) -> Dataset:
    """Muat korpus jsonl.gz menjadi satu Dataset.

    Args:
        files: Berkas korpus.
        columns: Nama kolom korpus; semuanya bertipe string.

    Returns:
        Dataset korpus dengan kolom sesuai columns.
    """
    features = Features({column: Value("string") for column in columns})
    return load_dataset("json", data_files=[str(file) for file in files], features=features, split="train")


corpus = load_corpus(corpus_files, CORPUS_COLUMNS)
print(corpus)

## 3. Topics dan qrels dev

Dimuat dengan builder `csv` berpemisah tab (K5). Berkas TSV tidak punya baris judul, jadi nama kolom ditetapkan di sel konstanta. Tanda kutip tidak diproses supaya teks query terbaca apa adanya.

In [ ]:
import csv


def load_tsv(path: Path, features: Features) -> Dataset:
    """Muat berkas TSV tanpa baris judul.

    Args:
        path: Berkas TSV.
        features: Nama dan tipe kolom, sesuai urutan kolom di berkas.

    Returns:
        Dataset berisi seluruh baris berkas.

    Raises:
        FileNotFoundError: Berkas tidak ada.
    """
    if not path.is_file():
        raise FileNotFoundError(f"Berkas TSV tidak ditemukan: {path}")
    return load_dataset(
        "csv",
        data_files=str(path),
        delimiter="\t",
        column_names=list(features),
        features=features,
        quoting=csv.QUOTE_NONE,
        split="train",
    )


topics_features = Features({column: Value("string") for column in TOPICS_COLUMNS})
qrels_features = Features(
    {
        "query_id": Value("string"),
        "iteration": Value("string"),
        "doc_id": Value("string"),
        "relevance": Value("int32"),
    }
)
topics = load_tsv(MIRACL_DIR / TOPICS_FILE, topics_features)
qrels = load_tsv(MIRACL_DIR / QRELS_FILE, qrels_features)
print(topics)
print(qrels)

## 4. Periksa jumlah baris

Jumlah baris harus sama dengan D3. Selisih berarti berkas yang dimuat bukan korpus id atau split dev yang dirancang.

In [ ]:
def validate_row_count(name: str, actual: int, expected: int) -> None:
    """Pastikan jumlah baris tabel sesuai D3.

    Args:
        name: Nama tabel untuk pesan error.
        actual: Jumlah baris yang dimuat.
        expected: Jumlah baris menurut D3.

    Raises:
        ValueError: Jumlah baris berbeda.
    """
    if actual != expected:
        raise ValueError(f"{name}: {actual} baris, seharusnya {expected} (D3)")


validate_row_count("corpus", corpus.num_rows, EXPECTED_CORPUS_ROWS)
validate_row_count("topics", topics.num_rows, EXPECTED_TOPICS_ROWS)
validate_row_count("qrels", qrels.num_rows, EXPECTED_QRELS_ROWS)
print({"corpus": corpus.num_rows, "topics": topics.num_rows, "qrels": qrels.num_rows})

## 5. Simpan ke `data/interim/`

Tabel disimpan sebagai Parquet dan metadata sebagai JSON (K9). Berkas ditimpa setiap notebook dijalankan karena isinya selalu turunan langsung dari `data/raw/`.

In [ ]:
import json
from datetime import datetime, timezone


def build_interim_metadata(sources: dict[str, dict[str, str | int | list[str]]]) -> dict[str, object]:
    """Susun isi metadata.json data interim.

    Args:
        sources: Per tabel: repo, revision, berkas sumber, dan jumlah baris.

    Returns:
        Metadata dengan cap waktu pembuatan.
    """
    # Cap waktu UTC format ISO 8601, contoh 2026-10-02T08:30:00+00:00
    created_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
    return {"created_at": created_at, "tables": sources}


def save_interim(tables: dict[str, Dataset], metadata: dict[str, object], interim_dir: Path) -> None:
    """Simpan tabel sebagai Parquet dan metadata sebagai JSON.

    Args:
        tables: Nama tabel ke Dataset; disimpan sebagai `<nama>.parquet`.
        metadata: Isi metadata.json.
        interim_dir: Folder tujuan.
    """
    interim_dir.mkdir(parents=True, exist_ok=True)
    for name, table in tables.items():
        table.to_parquet(str(interim_dir / f"{name}.parquet"))
    with (interim_dir / "metadata.json").open("w", encoding="utf-8", newline="\n") as file:
        json.dump(metadata, file, ensure_ascii=False, indent=2)


interim_metadata = build_interim_metadata(
    {
        "corpus": {"repo": CORPUS_REPO, "revision": corpus_revision, "files": corpus_paths, "rows": corpus.num_rows},
        "topics": {"repo": MIRACL_REPO, "revision": miracl_revision, "files": [TOPICS_FILE], "rows": topics.num_rows},
        "qrels": {"repo": MIRACL_REPO, "revision": miracl_revision, "files": [QRELS_FILE], "rows": qrels.num_rows},
    }
)
save_interim({"corpus": corpus, "topics": topics, "qrels": qrels}, interim_metadata, INTERIM_DIR)
print(sorted(path.name for path in INTERIM_DIR.iterdir()))